# Phase 4 — minimal configurable preprocessing
Raw covariates are the default. Optional standardization affects only x01–x06, using training means and population standard deviations. No oracle data are loaded. Upstream scaling does not imply a need to standardize again; this is an optional estimator requirement, not a selected improvement.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from causallens.data.loader import load_ihdp
from causallens.data.preprocess import IHDPPreprocessor, load_preprocessed_ihdp
root=Path.cwd()
if not (root/'pyproject.toml').exists(): root=root.parent
data_dir=root/'data/raw'
train=load_ihdp(0,'train',data_dir)
test=load_ihdp(0,'test',data_dir)
original={k:v.copy() for k,v in train.items()}
raw_train,raw_test=load_preprocessed_ihdp(0,data_dir=data_dir)
for a,b in ((train,raw_train),(test,raw_test)):
    for key in a:
        np.testing.assert_array_equal(a[key],b[key])
        assert a[key].dtype==b[key].dtype
        assert not np.shares_memory(a[key],b[key])
print('Raw mode preserves values and dtypes in independent arrays.')

In [ ]:
processor=IHDPPreprocessor(standardize=True)
scaled_train=processor.fit_transform(train)
scaled_test=processor.transform(test)
expected_mean=train['X'][:,:6].mean(axis=0)
expected_sd=train['X'][:,:6].std(axis=0,ddof=0)
np.testing.assert_allclose(processor.scaler_.mean_,expected_mean)
np.testing.assert_allclose(processor.scaler_.scale_,expected_sd)
np.testing.assert_allclose(scaled_test['X'][:,:6],(test['X'][:,:6]-expected_mean)/expected_sd)
np.testing.assert_allclose(scaled_train['X'][:,:6].mean(0),0,atol=1e-12)
np.testing.assert_allclose(scaled_train['X'][:,:6].std(0),1)
display(pd.DataFrame({'feature':[f'x{i:02d}' for i in range(1,7)],'raw_train_mean':expected_mean,'raw_train_sd':expected_sd,'scaled_train_mean':scaled_train['X'][:,:6].mean(0),'scaled_train_sd':scaled_train['X'][:,:6].std(0),'scaled_test_mean':scaled_test['X'][:,:6].mean(0)}))

In [ ]:
# Perturb a copy for an adversarial test; never rewrite benchmark files.
mean_before=processor.scaler_.mean_.copy()
scale_before=processor.scaler_.scale_.copy()
shifted={k:v.copy() for k,v in test.items()}
shifted['X'][:,:6]+=1000
shifted_output=processor.transform(shifted)
np.testing.assert_array_equal(processor.scaler_.mean_,mean_before)
np.testing.assert_array_equal(processor.scaler_.scale_,scale_before)
np.testing.assert_allclose(shifted_output['X'][:,:6]-scaled_test['X'][:,:6],np.broadcast_to(1000/scale_before,shifted_output['X'][:,:6].shape))
for a,b in ((train,scaled_train),(test,scaled_test)):
    np.testing.assert_array_equal(a['X'][:,6:],b['X'][:,6:])
    for key in ('T','Y'): np.testing.assert_array_equal(a[key],b[key])
for key in train: np.testing.assert_array_equal(train[key],original[key])
assert set(np.unique(scaled_train['X'][:,13]))=={1,2}
try:
    IHDPPreprocessor().fit({**train,'mu0':np.zeros(len(train['T']))})
except ValueError:
    print('Oracle field rejected as expected.')
else:
    raise AssertionError('Oracle rejection failed')
print('Training statistics fixed; binary encodings, T/Y and raw inputs preserved.')

In [ ]:
# Same estimator-facing dictionaries for either mode, independent replication.
for replication in (0,1):
    for standardize in (False,True):
        training,testing=load_preprocessed_ihdp(replication,standardize=standardize,data_dir=data_dir)
        assert set(training)==set(testing)=={'X','T','Y'}
        assert training['X'].shape==(672,25) and testing['X'].shape==(75,25)
print('Both modes verified on replications 0 and 1; official splits retained.')

## Interpretation
Test means need not be zero: all parameters came from training. No missing values require imputation; unexpected nonfinite inputs fail clearly. No outcomes are normalized. This notebook verifies implementation behavior, not estimator performance. If validation is later needed, split the official training partition first and fit only on that reduced training subset. No Phase 5 all-replication scientific QA is claimed.